<a href="https://colab.research.google.com/github/grasht/grashaw_GAN_research_project/blob/main/Diffusion_Model_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

# Generate a Tabular Dataset

In [11]:
data = torch.randn(1000, 5)

# Normalize (important for diffusion)
mean = data.mean(0)
std = data.std(0)
data = (data - mean) / std

# Hyper Parameters

In [12]:
time_steps = 1000

beta = torch.linspace(1e-4, 0.02, time_steps)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

# Diffusion Model (MLP)

In [13]:
class DiffusionModel(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, dim)
        )

    def forward(self, x, t):
        # Normalize timestep
        t = t.float().unsqueeze(1) / time_steps
        x = torch.cat([x, t], dim=1)
        return self.net(x)

In [14]:
model = DiffusionModel(dim=5)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

model.to(device)

cpu


DiffusionModel(
  (net): Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=5, bias=True)
  )
)

In [16]:
# Forward Diffusion

In [17]:
def forward_diffusion(x0, t):
    noise = torch.randn_like(x0)
    sqrt_alpha_bar = torch.sqrt(alpha_bar[t]).unsqueeze(1)
    sqrt_one_minus = torch.sqrt(1 - alpha_bar[t]).unsqueeze(1)

    xt = sqrt_alpha_bar * x0 + sqrt_one_minus * noise
    return xt, noise

# Training Loop

In [19]:
epochs = 10
batch_size = 64

for epoch in range(epochs):
    perm = torch.randperm(len(data))

    for i in range(0, len(data), batch_size):
        batch = data[perm[i:i+batch_size]]

        t = torch.randint(0, time_steps, (batch.size(0),))
        xt, noise = forward_diffusion(batch, t)

        pred_noise = model(xt, t)

        loss = nn.MSELoss()(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.5718
Epoch 2, Loss: 0.5190
Epoch 3, Loss: 0.4005
Epoch 4, Loss: 0.2368
Epoch 5, Loss: 0.3415
Epoch 6, Loss: 0.2545
Epoch 7, Loss: 0.3003
Epoch 8, Loss: 0.3053
Epoch 9, Loss: 0.4567
Epoch 10, Loss: 0.3737


# Data Generation



In [21]:
@torch.no_grad()
def sample(model, n_samples, dim):
    x = torch.randn(n_samples, dim)

    for t in reversed(range(time_steps)):
        t_tensor = torch.full((n_samples,), t)

        pred_noise = model(x, t_tensor)

        a = alpha[t]
        a_bar = alpha_bar[t]
        b = beta[t]

        if t > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (1 / torch.sqrt(a)) * (
            x - (1 - a) / torch.sqrt(1 - a_bar) * pred_noise
        ) + torch.sqrt(b) * noise

    return x


# Generate synthetic data
samples = sample(model, n_samples=10, dim=5)

# Denormalize
samples = samples * std + mean

print("Generated Samples:")
print(samples)

Generated Samples:
tensor([[ 0.2429, -0.1087,  0.3278,  2.1262,  0.2218],
        [ 0.5648,  0.5895,  0.7547,  0.0958, -0.8157],
        [ 1.1482,  0.6357,  0.1498,  0.5505,  0.1535],
        [ 0.3285,  0.0884,  0.0382, -0.7778,  0.6807],
        [ 1.7586,  1.0929,  0.0311, -1.1629, -0.0249],
        [ 0.9265,  0.3652, -0.0818, -0.4792, -0.2243],
        [-0.4021,  0.6932,  0.9080,  1.2950,  1.3439],
        [ 0.5407, -0.1836, -0.0795,  0.1383,  1.0982],
        [-0.1058,  0.4258, -0.2396, -0.0042,  0.0371],
        [ 1.1080, -0.0203, -0.4528, -0.4509,  0.5684]])
